# Homework 1 (Module 1 — 2026 Cohort)
**Course:** [Stock Markets Analytics Zoomcamp](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp)

This notebook contains complete interactive solutions for Module 1 Homework questions.

In [1]:
import io
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Set display preferences
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', None)

## Question 1: S&P 500 Stocks Added to the Index

- Download the list of S&P 500 companies from Wikipedia.
- Find which year (from 2020 onwards) had the largest number of new companies added to the index.

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(url, headers=headers)
tables = pd.read_html(io.StringIO(response.text))
sp500_table = tables[0]

# Clean and parse Date added
sp500_table["Date added"] = pd.to_datetime(sp500_table["Date added"], errors="coerce")
sp500_table["Year added"] = sp500_table["Date added"].dt.year

additions_from_2020 = (
    sp500_table[sp500_table["Year added"] >= 2020]["Year added"]
    .value_counts()
    .sort_index()
)

print("Number of additions per year (>= 2020):")
for year, count in additions_from_2020.items():
    print(f"  {int(year)}: {count} companies")

# Full years evaluation (2020-2025)
full_years = additions_from_2020[additions_from_2020.index <= 2025]
max_year = int(full_years.idxmax())
max_count = int(full_years.max())

print(f"\n Answer Q1: {max_year} had the highest number of additions ({max_count} stocks).")

# Additional stat: > 20 years in index
older_than_20y = sp500_table[sp500_table["Date added"] < "2006-01-01"]
print(f" Additional: {len(older_than_20y)} stocks have been in the index for > 20 years.")

Number of additions per year (>= 2020):
  2020: 10 companies
  2021: 10 companies
  2022: 15 companies
  2023: 15 companies
  2024: 16 companies
  2025: 18 companies
  2026: 13 companies

 Answer Q1: 2025 had the highest number of additions (18 stocks).
 Additional: 218 stocks have been in the index for > 20 years.


## Question 2: Macro Indexes YTD Performance (as of 21 August 2026)

- Compare S&P 500 YTD return against 10 major global indexes:
  - China (Shanghai Composite: `000001.SS`)
  - Hong Kong (Hang Seng: `^HSI`)
  - Australia (S&P/ASX 200: `^AXJO`)
  - India (Nifty 50: `^NSEI`)
  - Canada (S&P/TSX Composite: `^GSPTSE`)
  - Germany (DAX: `^GDAXI`)
  - UK (FTSE 100: `^FTSE`)
  - Japan (Nikkei 225: `^N225`)
  - Mexico (IPC Mexico: `^MXX`)
  - Brazil (Ibovespa: `^BVSP`)
- Find how many indexes outperformed the S&P 500 YTD.

In [3]:
indexes = {
    "United States - S&P 500": "^GSPC",
    "China - Shanghai Composite": "000001.SS",
    "Hong Kong - HANG SENG INDEX": "^HSI",
    "Australia - S&P/ASX 200": "^AXJO",
    "India - Nifty 50": "^NSEI",
    "Canada - S&P/TSX Composite": "^GSPTSE",
    "Germany - DAX": "^GDAXI",
    "United Kingdom - FTSE 100": "^FTSE",
    "Japan - Nikkei 225": "^N225",
    "Mexico - IPC Mexico": "^MXX",
    "Brazil - Ibovespa": "^BVSP",
}

returns = {}
print("Downloading index data (2026-01-01 to 2026-08-22)...")
for name, ticker in indexes.items():
    df = yf.download(ticker, start="2026-01-01", end="2026-08-22", progress=False)
    if not df.empty and len(df) > 1:
        close = df["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        close = close.dropna()
        first_val = close.iloc[0]
        last_val = close.iloc[-1]
        ytd = (last_val - first_val) / first_val * 100
        returns[name] = ytd
        print(f"  {name:32s} ({ticker:10s}): {ytd:+.2f}%")

us_return = returns["United States - S&P 500"]
outperforming = {
    k: v for k, v in returns.items() if k != "United States - S&P 500" and v > us_return
}

print(f"\nS&P 500 Return: {us_return:+.2f}%")
print(f" Answer Q2: {len(outperforming)} indexes (out of 10) have better YTD returns than S&P 500.")
for name, val in outperforming.items():
    print(f"   * {name}: {val:+.2f}%")

  United States - S&P 500          (^GSPC     ): +11.90%
  China - Shanghai Composite       (000001.SS ): -2.94%
  Hong Kong - HANG SENG INDEX      (^HSI      ): -1.25%
  Australia - S&P/ASX 200          (^AXJO     ): +3.79%
  India - Nifty 50                 (^NSEI     ): -7.25%
  Canada - S&P/TSX Composite       (^GSPTSE   ): +14.86%
  Germany - DAX                    (^GDAXI    ): +6.51%
  United Kingdom - FTSE 100        (^FTSE     ): +8.70%
  Japan - Nikkei 225               (^N225     ): +27.36%
  Mexico - IPC Mexico              (^MXX      ): +2.48%
  Brazil - Ibovespa                (^BVSP     ): +6.54%

S&P 500 Return: +11.90%
 Answer Q2: 2 indexes (out of 10) have better YTD returns than S&P 500.
   * Canada - S&P/TSX Composite: +14.86%
   * Japan - Nikkei 225: +27.36%


## Question 3: S&P 500 Market Corrections Analysis (1950 - Present)

- Download historical daily data for `^GSPC` from 1950.
- Identify market corrections (drawdowns $\ge 5\%$) between all-time highs (ATH).
- Calculate median drawdown and duration.

In [4]:
df = yf.download("^GSPC", start="1950-01-01", progress=False)
close = df["Close"]
if isinstance(close, pd.DataFrame):
    close = close.iloc[:, 0]
close = close.dropna()

running_max = close.cummax()
is_ath = close == running_max
ath_dates = close[is_ath].index

corrections = []
for i in range(len(ath_dates) - 1):
    start_ath = ath_dates[i]
    end_ath = ath_dates[i + 1]

    period = close.loc[start_ath:end_ath]
    if len(period) <= 1:
        continue

    min_price = period.min()
    min_date = period.idxmin()
    high_price = close.loc[start_ath]

    drawdown_pct = (high_price - min_price) / high_price * 100
    duration_days = (min_date - start_ath).days  # Days to trough (matches Hint)
    duration_recovery_days = (end_ath - start_ath).days  # Full cycle to next ATH

    if drawdown_pct >= 5.0:
        corrections.append(
            {
                "start_ath": start_ath.strftime("%Y-%m-%d"),
                "end_ath": end_ath.strftime("%Y-%m-%d"),
                "min_date": min_date.strftime("%Y-%m-%d"),
                "drawdown_pct": drawdown_pct,
                "duration_days": duration_days,
                "duration_recovery_days": duration_recovery_days,
            }
        )

corr_df = pd.DataFrame(corrections)
p25_dd = np.percentile(corr_df["drawdown_pct"], 25)
p50_dd = np.percentile(corr_df["drawdown_pct"], 50)
p75_dd = np.percentile(corr_df["drawdown_pct"], 75)

p25_dur = np.percentile(corr_df["duration_days"], 25)
p50_dur = np.percentile(corr_df["duration_days"], 50)
p75_dur = np.percentile(corr_df["duration_days"], 75)

p25_rec = np.percentile(corr_df["duration_recovery_days"], 25)
p50_rec = np.percentile(corr_df["duration_recovery_days"], 50)
p75_rec = np.percentile(corr_df["duration_recovery_days"], 75)

print(f"Total corrections identified (>= 5%): {len(corr_df)}")
print(f"-> Drawdowns (%): 25th={p25_dd:.2f}%, Median (50th)={p50_dd:.2f}%, 75th={p75_dd:.2f}%")
print(f"-> Duration to Trough (days): 25th={p25_dur:.1f}d, Median (50th)={p50_dur:.1f}d, 75th={p75_dur:.1f}d")
print(f"-> Full Cycle to next ATH (days): 25th={p25_rec:.1f}d, Median (50th)={p50_rec:.1f}d, 75th={p75_rec:.1f}d")
print(f" Answer Q3: Median drawdown is {p50_dd:.2f}% (median duration to trough is {p50_dur:.1f} days, full cycle is {p50_rec:.1f} days).")

corr_df


Total corrections identified (>= 5%): 74
-> Drawdowns (%): 25th=6.23%, Median (50th)=7.99%, 75th=14.02%
-> Duration to Trough (days): 25th=22.0d, Median (50th)=40.5d, 75th=86.2d
-> Full Cycle to next ATH (days): 25th=55.5d, Median (50th)=92.5d, 75th=211.5d
 Answer Q3: Median drawdown is 7.99% (median duration to trough is 40.5 days, full cycle is 92.5 days).


,start_ath,end_ath,min_date,drawdown_pct,duration_days,duration_recovery_days
0,1950-06-12,1950-09-22,1950-07-17,14.02,35,102
1,1950-11-24,1950-12-28,1950-12-04,6.50,10,34
2,1951-05-03,1951-08-02,1951-06-29,8.11,57,91
3,1951-10-15,1952-01-03,1951-11-23,6.08,39,80
4,1952-01-22,1952-06-25,1952-02-20,6.37,29,155
5,1952-08-08,1952-11-28,1952-10-22,6.85,75,112
6,1953-01-05,1954-03-11,1953-09-14,14.82,252,430
7,1955-01-03,1955-02-04,1955-01-17,5.90,14,32
8,1955-03-04,1955-04-12,1955-03-14,6.82,10,39
9,1955-09-23,1955-11-14,1955-10-11,10.59,18,52


## Question 4: Earnings Surprise Analysis for Amazon (AMZN)

- Download earnings surprise history and price history for AMZN.
- Calculate 2-day return after positive earnings surprise announcements.
- Find the median 2-day return and correlation with surprise magnitude.

In [5]:
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)
earnings_df = ticker_obj.get_earnings_dates()

price_df = yf.download(ticker, start="2020-01-01", progress=False)
close = price_df["Close"]
if isinstance(close, pd.DataFrame):
    close = close.iloc[:, 0]

earnings_clean = earnings_df.dropna(subset=["Surprise(%)"]).copy()
pos_surprises = earnings_clean[earnings_clean["Surprise(%)"] > 0].copy()
returns_2d = []
surprises = []

close_dates = (
    close.index.tz_localize(None).normalize()
    if close.index.tz is not None
    else close.index.normalize()
)

for date_idx, row in pos_surprises.iterrows():
    dt = (
        date_idx.tz_localize(None).normalize()
        if hasattr(date_idx, "tz_localize") and date_idx.tz is not None
        else pd.Timestamp(date_idx).normalize()
    )
    surprise = row["Surprise(%)"]

    matches = np.where(close_dates >= dt)[0]
    if len(matches) > 0:
        day2_idx = matches[0]
        day1_idx = day2_idx - 1
        day3_idx = day2_idx + 1

        if day1_idx >= 0 and day3_idx < len(close):
            c1 = close.iloc[day1_idx]
            c3 = close.iloc[day3_idx]
            ret_2d = (c3 / c1 - 1) * 100
            returns_2d.append(ret_2d)
            surprises.append(surprise)
# returns_2d
res_df = pd.DataFrame({"Surprise(%)": surprises, "2d_Return(%)": returns_2d})
median_ret = res_df["2d_Return(%)"].median()
corr = res_df.corr().iloc[0, 1]

print(f"Positive surprise events analyzed: {len(res_df)}")
print(f" Answer Q4: Median 2-day return after positive surprise is {median_ret:.2f}%.")
print(f"   Correlation between surprise magnitude and 2-day return: {corr:.4f}")

res_df

Positive surprise events analyzed: 20
 Answer Q4: Median 2-day return after positive surprise is 0.35%.
   Correlation between surprise magnitude and 2-day return: 0.3306


,Surprise(%),2d_Return(%)
0,215.02,19.82
1,69.02,2.06
2,0.22,-9.73
3,25.20,6.04
4,27.19,-6.71
5,16.77,3.01
6,25.29,-2.97
7,25.22,2.70
8,23.76,-10.20
9,17.67,-1.08


## Q5: Capstone Project Idea
**Title:** Market Dashboard: Smart System for Spotting Market Trends and Risk

**Executive Summary:** This project builds an automated tool that checks the health of the US stock market and warns about potential downturns 2 to 6 weeks in advance. It combines major stock indexes, market breadth, and economic indicators using machine learning.

**Key Data Layers:**
* Main Indexes: SPY, QQQ, DIA, IWM, and SMH (semiconductors as a lead indicator).
* Market Health: How many stocks are actually rising vs. falling (using EMAs, McClellan Oscillator, and RSP/SPY ratio).
* Macro & Risk: Treasury yields, the US Dollar, VIX (fear index), and option put/call ratios.
* Technical Signals: Moving averages across multiple timeframes and Bollinger/Fibonacci channels to spot trend shifts.

**How Machine Learning Is Used:**
* Find Market States: Uses clustering models (HMM/GMM) to group the market into 3 simple modes: Bullish, Caution, or High Risk.
* Predict Drops: Uses tree-based models (XGBoost/LightGBM) to estimate the chance of a 3–5% drop in the next 15 days.
* Traffic Light Signal: Converts complex data into a simple actionable output: Green (Buy/Hold), Yellow (Be Careful), or Red (Move to Cash/Hedge).

## Q6: Metrics & Key Project Elements

Market Health Metrics: Percent of stocks above key moving averages (20, 50, 100, 200 EMAs) and net new highs vs. lows.
Market Ratios: SMH / SPY to track tech risk-taking, and RSP / SPY to measure broad stock participation.
Macro & Volatility: 20-day yield momentum, US Dollar trends, and VIX/VVIX spikes to track market stress.
Backtest Evaluation: Strategy performance vs. SPY (2005–2026) focusing on Sharpe, Sortino, and Calmar ratios, alongside overall Max Drawdown reduction.
Final Output: An interactive Streamlit dashboard with real-time gauges and automated market status updates.